# 🚨 Fintra-AI: Anomaly, Outlier & Duplicate Analysis (03_anomaly_analysis.ipynb)

### 🎯 Objective
This notebook implements robust statistical methods to identify:
1. **Unusual Transaction Amounts**: Extreme monetary values using Interquartile Range (IQR) and Z-score methods.
2. **Duplicate & Double-Charge Records**: Multiple charges with matching merchant and amount within short time windows.
3. **Behavioral Spending Pattern Anomalies**: Transactions deviating significantly from category baselines, flagged with explainable diagnostic reason codes.

> **Important**: This analysis identifies statistical anomalies for budget auditing and user awareness. It strictly avoids labelling transactions as fraudulent without external fraud ground-truth.


In [1]:
# 1. Imports and Setup
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath(".."))

from ml.analysis.data_loader import load_project_dataset, generate_sample_financial_dataset
from ml.analysis.anomaly_analyzer import (
    detect_amount_outliers_iqr,
    detect_amount_outliers_zscore,
    detect_duplicate_transactions,
    analyze_unexpected_spending_patterns
)

# Load data
df = load_project_dataset(include_raw_sources=True)
if df.empty:
    df = generate_sample_financial_dataset(n_records=400, seed=42)

print(f"Loaded {len(df)} transactions for anomaly analysis.")


Loaded 10440 transactions for anomaly analysis.


## 1. Outlier Detection: IQR vs. Z-Score Methods

* **IQR Method**: Robust against skewed financial distributions. $IQR = Q_3 - Q_1$, Outliers $> Q_3 + 1.5 \times IQR$.
* **Z-Score Method**: Standard deviations from the mean ($|Z| \ge 3.0$).


In [2]:
# Category-specific IQR Outlier Detection
iqr_outliers = detect_amount_outliers_iqr(df, group_by_category=True, iqr_multiplier=1.5)
print(f"Total Category-level IQR Outliers Flagged: {len(iqr_outliers)}")
print(iqr_outliers[["date", "merchant", "category", "amount", "iqr_upper_bound", "deviation_from_median"]].head(10))


Total Category-level IQR Outliers Flagged: 688
                 date               merchant  category     amount  \
0 2017-12-26 21:55:12          Fixed Deposit     other  250000.00   
1 2017-06-27 10:00:50          Fixed Deposit     other  200000.00   
2 2017-12-26 21:50:33  Saving Bank account 1     other  150000.00   
3 2017-10-10 00:00:00           Share Market     other  150000.00   
4 2025-01-21 00:00:00            Apple Store  shopping  149925.20   
5 2025-05-26 00:00:00            Apple Store  shopping  149836.10   
6 2025-05-28 00:00:00         Dell Exclusive  shopping  149827.58   
7 2025-11-08 00:00:00                  Noise  shopping  149809.51   
8 2025-08-29 00:00:00                OnePlus  shopping  149807.19   
9 2025-04-24 00:00:00               HP World  shopping  149737.72   

   iqr_upper_bound  deviation_from_median  
0         12125.00              249000.00  
1         12125.00              199000.00  
2         12125.00              149000.00  
3         12125.0

In [3]:
# Global Z-Score Outliers
z_outliers = detect_amount_outliers_zscore(df, threshold=3.0)
print(f"Total Z-Score Extreme Outliers (|Z| >= 3.0): {len(z_outliers)}")
print(z_outliers[["date", "merchant", "category", "amount", "z_score"]].head(10))


Total Z-Score Extreme Outliers (|Z| >= 3.0): 388
                 date               merchant  category     amount  z_score
0 2017-12-26 21:55:12          Fixed Deposit     other  250000.00    10.02
1 2017-06-27 10:00:50          Fixed Deposit     other  200000.00     7.93
2 2017-10-10 00:00:00           Share Market     other  150000.00     5.84
3 2025-01-21 00:00:00            Apple Store  shopping  149925.20     5.84
4 2017-12-26 21:50:33  Saving Bank account 1     other  150000.00     5.84
5 2025-11-08 00:00:00                  Noise  shopping  149809.51     5.83
6 2025-05-28 00:00:00         Dell Exclusive  shopping  149827.58     5.83
7 2025-04-24 00:00:00               HP World  shopping  149737.72     5.83
8 2025-08-29 00:00:00                OnePlus  shopping  149807.19     5.83
9 2025-05-26 00:00:00            Apple Store  shopping  149836.10     5.83


## 2. Duplicate Transaction & Double-Charge Audit

Detects transactions occurring at the same merchant for the same amount within a 24-hour window.


In [4]:
# Duplicate Audit (Flagged for Review)
duplicates = detect_duplicate_transactions(df, time_window_hours=24.0)
print(f"Potential Duplicate Transactions Flagged for Review: {len(duplicates)}")
print(duplicates)


Potential Duplicate Transactions Flagged for Review: 211
            merchant   category   amount       original_date  \
0               auto  transport     30.0 2017-11-19 20:22:19   
1             snacks       food    150.0 2018-03-22 19:31:16   
2               Taxi  transport     30.0 2017-03-31 21:06:32   
3    Maturity amount      other  40326.0 2017-07-27 05:39:02   
4               auto  transport     30.0 2017-01-29 19:22:36   
..               ...        ...      ...                 ...   
206             auto  transport     15.0 2018-05-07 00:00:00   
207             auto  transport     60.0 2017-11-08 00:00:00   
208        breakfast       food     35.0 2017-08-05 00:00:00   
209             maid      bills   1000.0 2017-01-05 00:00:00   
210             maid      bills   2000.0 2018-01-01 00:00:00   

    potential_duplicate_date  time_gap_hours  \
0        2017-11-19 20:23:06            0.01   
1        2018-03-22 19:31:37            0.01   
2        2017-03-31 21:07:22  

## 3. Explainable Behavioral Spending Anomaly Diagnostics

Each flagged transaction is assigned a human-readable reason code explaining why it was flagged (e.g., $5\times$ category median, late night off-hours).


In [5]:
# Behavioral Spending Pattern Analysis
pattern_anomalies = analyze_unexpected_spending_patterns(df)
print(f"Transactions Flagged with Behavioral Diagnostics: {len(pattern_anomalies)}")
print(pattern_anomalies[["date", "merchant", "category", "amount", "spend_ratio", "severity", "diagnostic_reasons"]].head(12))


Transactions Flagged with Behavioral Diagnostics: 9183
                  date               merchant  category     amount  \
0  2017-12-26 21:55:12          Fixed Deposit     other  250000.00   
1  2017-06-27 10:00:50          Fixed Deposit     other  200000.00   
2  2017-10-10 00:00:00           Share Market     other  150000.00   
3  2017-12-26 21:50:33  Saving Bank account 1     other  150000.00   
4  2025-01-21 00:00:00            Apple Store  shopping  149925.20   
5  2025-05-26 00:00:00            Apple Store  shopping  149836.10   
6  2025-05-28 00:00:00         Dell Exclusive  shopping  149827.58   
7  2025-11-08 00:00:00                  Noise  shopping  149809.51   
8  2025-08-29 00:00:00                OnePlus  shopping  149807.19   
9  2025-04-24 00:00:00               HP World  shopping  149737.72   
10 2025-07-14 00:00:00                  Croma  shopping  149593.13   
11 2025-10-13 00:00:00                  Noise  shopping  149349.96   

    spend_ratio severity          